# 🏥 Disease Prediction - Data Exploration & Model Analysis

This notebook walks you through:
1. Understanding your dataset
2. Exploring symptoms and diseases
3. Visualizing the data
4. Understanding why we chose the best model

**Run `train_model.py` before this notebook to train the model.**

In [ ]:
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

sns.set_theme(style='whitegrid', palette='husl')
print('Libraries loaded ✅')

## 1. Load All Data Files

In [ ]:
# Load training data (it's stored as a nested zip)
with zipfile.ZipFile('../data/Training.csv', 'r') as z:
    with z.open('Training.csv') as f:
        df = pd.read_csv(f)

desc    = pd.read_csv('../data/description.csv')
prec    = pd.read_csv('../data/precautions_df.csv')
meds    = pd.read_csv('../data/medications.csv')
diets   = pd.read_csv('../data/diets.csv')
workout = pd.read_csv('../data/workout_df.csv')
severity = pd.read_csv('../data/Symptom-severity.csv')

print(f'Training data: {df.shape}')
print(f'Diseases     : {df["prognosis"].nunique()}')
print(f'Symptoms     : {df.shape[1]-1}')
df.head()

## 2. Disease Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
disease_counts = df['prognosis'].value_counts()
disease_counts.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Number of Records per Disease', fontsize=14)
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()
print('Dataset is balanced:', disease_counts.std() < 10)

## 3. Symptom Severity Weights

In [ ]:
top_severe = severity.sort_values('weight', ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=top_severe, x='weight', y='Symptom', ax=ax, palette='Reds_r')
ax.set_title('Top 20 Most Severe Symptoms')
plt.tight_layout()
plt.show()

## 4. Symptom Co-occurrence Heatmap (top 20 symptoms)

In [ ]:
X = df.drop('prognosis', axis=1)
top_symptoms = X.sum().nlargest(20).index
corr = X[top_symptoms].corr()

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr, cmap='coolwarm', ax=ax, annot=False, linewidths=0.3)
ax.set_title('Symptom Co-occurrence Correlation (Top 20 Symptoms)')
plt.tight_layout()
plt.show()

## 5. Read Model Report

In [ ]:
with open('../models/model_report.txt', 'r') as f:
    print(f.read())

## 6. Test the Loaded Model

In [ ]:
with open('../models/best_model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('../models/label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)
with open('../models/feature_columns.pkl', 'rb') as f:
    feature_columns = pickle.load(f)

# Predict for test symptoms
test_symptoms = ['fever', 'headache', 'nausea', 'vomiting']

input_vector = np.zeros(len(feature_columns))
col_map = {c.strip().lower(): i for i, c in enumerate(feature_columns)}
for sym in test_symptoms:
    key = sym.strip().lower().replace(' ', '_')
    if key in col_map:
        input_vector[col_map[key]] = 1

pred = model.predict(input_vector.reshape(1, -1))[0]
disease = le.inverse_transform([pred])[0]

if hasattr(model, 'predict_proba'):
    proba = model.predict_proba(input_vector.reshape(1, -1))[0]
    confidence = proba[pred] * 100
    print(f'Symptoms  : {test_symptoms}')
    print(f'Prediction: {disease}')
    print(f'Confidence: {confidence:.1f}%')
else:
    print(f'Prediction: {disease}')

## 7. Feature Importance (if Random Forest is best model)

In [ ]:
# Only works if the best model is Random Forest
try:
    importances = model.feature_importances_
    top_idx = np.argsort(importances)[-20:]
    top_features = [feature_columns[i] for i in top_idx]
    top_scores = importances[top_idx]

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(x=top_scores, y=top_features, ax=ax, palette='viridis')
    ax.set_title('Top 20 Most Important Symptoms (Random Forest)')
    plt.tight_layout()
    plt.show()
except AttributeError:
    print('Feature importance not available for this model type.')
    print('(Only available for tree-based models like Random Forest)')